# Petasos — Live Routing Demo

Run every cell (Runtime -> Run all). This notebook downloads the real baked-in
**Qwen2.5-3B-Instruct** GGUF and runs the **real** `router.py` / `confidence.py`
routing and escalation-gate code from the scored competition container (synced
copies, not a reimplementation) — nothing here is mocked.

For each prompt you'll see: the category it's routed to (Stage-1 heuristic),
the difficulty estimate, the local model's answer, and whether the escalation
gate would send it to Fireworks. **No live Fireworks call is made** — an
"escalate" verdict is reported but not executed, so this public notebook can't
spend the project's remote token budget.

Full project, README, and Dockerfile: https://github.com/marios-gasparis/petasos-routing-agent


In [ ]:
!pip -q install llama-cpp-python huggingface_hub ipywidgets


In [ ]:
from huggingface_hub import hf_hub_download

MODEL_PATH = hf_hub_download(repo_id="Qwen/Qwen2.5-3B-Instruct-GGUF", filename="qwen2.5-3b-instruct-q4_k_m.gguf")
print(MODEL_PATH)


In [ ]:
%%writefile prompts.py
"""Terse per-category system prompts. Kept separate from categories.py so
prompt wording can be tuned without touching max_tokens/stop/needs_cot."""

SYSTEM_PROMPTS = {
    "factual": (
        "Answer the question factually and concisely. Answer only, no preamble."
    ),
    "math": (
        "Solve the problem. Show brief work only if needed, then end with the "
        "final numeric answer on its own line."
    ),
    "sentiment": (
        "Classify the sentiment as exactly one word: positive, negative, or "
        "neutral. Answer only, no explanation."
    ),
    "summarization": (
        "Summarize the text in 1-3 concise sentences. Answer only, no preamble."
    ),
    "ner": (
        "Extract all named entities from the text. Return them as a JSON list "
        "of strings, nothing else."
    ),
    "code-debug": (
        "Find and fix the bug in the code. Return only the corrected code."
    ),
    "logic": (
        "Solve the logic problem. Reason briefly step by step, then end with "
        "the final answer on its own line."
    ),
    "code-gen": (
        "Write the requested code. Return only the code, no explanation."
    ),
}


In [ ]:
%%writefile categories.py
"""The 8 task categories and their per-category generation settings."""

from dataclasses import dataclass
from typing import List, Optional

from prompts import SYSTEM_PROMPTS

# Starting max_tokens caps (compass_artifact plan, PHASE_3).
MAX_TOKENS = {
    "factual": 40,
    "math": 256,
    "sentiment": 5,
    "summarization": 120,
    "ner": 60,
    "code-debug": 300,
    "logic": 256,
    "code-gen": 400,
}

NEEDS_COT = {"math", "logic"}

DEFAULT_CATEGORY = "factual"


@dataclass(frozen=True)
class Category:
    name: str
    system_prompt: str
    max_tokens: int
    stop: Optional[List[str]] = None
    needs_cot: bool = False


CATEGORIES = {
    name: Category(
        name=name,
        system_prompt=SYSTEM_PROMPTS[name],
        max_tokens=MAX_TOKENS[name],
        needs_cot=name in NEEDS_COT,
    )
    for name in SYSTEM_PROMPTS
}


def get_category(name):
    """Look up a Category, falling back to the default for unknown names."""
    return CATEGORIES.get(name, CATEGORIES[DEFAULT_CATEGORY])


In [ ]:
%%writefile router.py
"""Two-stage router.

Stage 1 (this module's `heuristic_route`): keyword/regex classification into
one of the 8 categories plus a cheap difficulty estimate. It is the proven
backbone (100% category accuracy on the dev set) and the always-available
fallback.

Stage 2 (Phase 6): a lightweight learned classifier (TF-IDF + logistic
regression, loaded via `classifier_router`). When enabled and confident, it can
override the heuristic's category; otherwise routing defers to Stage 1.

Stage 2 is DISABLED BY DEFAULT (env `ROUTER_USE_CLASSIFIER=1` to enable). On the
offline testset the heuristic already scores 100% on held-out variants and the
category head does not beat it, so -- per CLAUDE.md's "never sacrifice accuracy"
mandate -- the classifier is kept off until a live harness run confirms it
helps (and until the predicted-local-success head, which is the real
token-saving knob, is trained on Phase 5 labels). The wiring below is complete
so enabling it is a one-env-var flip. Difficulty is ALWAYS taken from the
heuristic (the classifier does not predict difficulty)."""

import logging
import os
import re

from categories import DEFAULT_CATEGORY, get_category

logger = logging.getLogger(__name__)


def _classifier_enabled():
    """Read at call time (like config.resolve_models) so it reflects the
    container's live environment. Default off."""
    return os.environ.get("ROUTER_USE_CLASSIFIER", "0").strip().lower() in (
        "1", "true", "yes", "on")

_CODE_BLOCK_RE = re.compile(r"```")
_BUG_RE = re.compile(r"\b(fix|bug|error|debug|traceback|exception|doesn't work|not working)\b", re.I)
# Allow adjectives between the verb and the noun ("write a Python function",
# "create a recursive method") -- the old literal "write a function" missed
# every real-world phrasing that named a language. Bounded gap ([^.?!]{0,40})
# so it stays within a single clause and can't span sentences.
_CODEGEN_RE = re.compile(
    r"\b(write|create|generate|implement)\b[^.?!]{0,40}?"
    r"\b(function|program|script|method|class|algorithm|code|snippet)\b"
    r"|\bimplement\b",
    re.I,
)
_NER_RE = re.compile(r"\b(extract .*(entities|entity)|named entit(y|ies))\b", re.I)
_SUMMARY_RE = re.compile(r"\b(summarize|summari[sz]ation|tl;?dr)\b", re.I)
# Bare "classify" is treated as sentiment: within these 8 categories it is the
# only classification task, and this branch runs after ner/summarization so it
# can't steal those.
_SENTIMENT_RE = re.compile(r"\b(sentiment|positive or negative|is this review|classify)\b", re.I)
_MATH_KEYWORD_RE = re.compile(
    r"\b(calculate|compute|how many|how much|sum of|product of|solve for|"
    r"average of|total of|total cost|percent|percentage|square root|cube root|"
    r"multiplied by|divided by)\b",
    re.I,
)
_NUMBER_RE = re.compile(r"-?\d+(\.\d+)?")
_OPERATOR_RE = re.compile(r"[+\-*/^%=]|\bplus\b|\bminus\b|\btimes\b|\bdivided by\b")
# Beyond explicit connectives, catch relational/deductive phrasings that read
# like factual questions: comparatives ("older than"), spatial relations
# ("north of"), quantified statements, and yes/no deductions. Deliberately no
# bare superlatives (e.g. "largest") -- those collide with factual questions
# like "the largest planet"; rely on the relational "... than" / "furthest"
# forms instead.
_LOGIC_RE = re.compile(
    r"\b(therefore|deduce|syllogism|puzzle|"
    r"if .+ then|either .+ or|all \w+ are|no \w+s? (can|are|is)|every .+ (is|are)|"
    r"(older|younger|taller|shorter|bigger|smaller|faster|slower|heavier|lighter|"
    r"greater|higher|lower|more|fewer|closer|further|farther) than|"
    r"north of|south of|east of|west of|furthest|farthest|closest|"
    r"answer yes or no|what day)\b",
    re.I,
)
_REASONING_CUES_RE = re.compile(r"\b(why|explain|step by step|first.*then|after that|because)\b", re.I)

_HARD_LENGTH_THRESHOLD = 300
_HARD_NUMBER_COUNT = 3


def heuristic_route(prompt):
    """Classify a prompt into (category, difficulty).

    Signal precedence deliberately puts code detection before math: code
    snippets routinely contain digits and operators that would otherwise be
    misread as a math prompt.
    """
    if not prompt or not prompt.strip():
        return DEFAULT_CATEGORY, "easy"

    text = prompt.strip()
    lower = text.lower()
    has_code_block = bool(_CODE_BLOCK_RE.search(text))

    if has_code_block and _BUG_RE.search(lower):
        category = "code-debug"
    elif _CODEGEN_RE.search(lower):
        category = "code-gen"
    elif has_code_block:
        # A bare code block with no bug/codegen wording is most often a
        # debugging request ("here's my code, what's wrong").
        category = "code-debug"
    elif _NER_RE.search(lower):
        category = "ner"
    elif _SUMMARY_RE.search(lower):
        category = "summarization"
    elif _SENTIMENT_RE.search(lower):
        category = "sentiment"
    elif _MATH_KEYWORD_RE.search(lower) or (_NUMBER_RE.search(text) and _OPERATOR_RE.search(lower)):
        category = "math"
    elif _LOGIC_RE.search(lower):
        category = "logic"
    else:
        category = DEFAULT_CATEGORY

    difficulty = _estimate_difficulty(text, lower, category)
    return category, difficulty


def _estimate_difficulty(text, lower, category):
    if len(text) > _HARD_LENGTH_THRESHOLD:
        return "hard"

    if category in ("math", "logic"):
        number_count = len(_NUMBER_RE.findall(text))
        if number_count >= _HARD_NUMBER_COUNT or _REASONING_CUES_RE.search(lower):
            return "hard"

    if category in ("code-debug", "code-gen"):
        if text.count("```") > 2 or len(text) > 150:
            return "hard"

    if _REASONING_CUES_RE.search(lower):
        return "hard"

    return "easy"


def route_task(prompt):
    """Resolve (Category object, category_name, difficulty) for a prompt.

    Category comes from Stage 2 (the learned classifier) when it is enabled AND
    confident above its tuned gate; otherwise from the Stage-1 heuristic.
    Difficulty is always the heuristic's estimate. Public shape is unchanged so
    main.py and the harness need no rewrite."""
    heuristic_category, difficulty = heuristic_route(prompt)
    category_name = heuristic_category

    if _classifier_enabled():
        category_name = _stage2_category(prompt, heuristic_category)

    return get_category(category_name), category_name, difficulty


def _stage2_category(prompt, heuristic_category):
    """Return the classifier's category if it is available and its confidence
    clears the tuned gate; otherwise the heuristic's. Any failure inside the
    classifier path degrades silently to the heuristic (never raises)."""
    try:
        import classifier_router

        result = classifier_router.classify_category(prompt)
        if result is None:
            return heuristic_category
        clf_category, confidence = result
        if confidence >= classifier_router.min_confidence():
            if clf_category != heuristic_category:
                logger.info(
                    "Stage-2 override: heuristic=%s -> classifier=%s (conf=%.3f)",
                    heuristic_category, clf_category, confidence,
                )
            return clf_category
    except Exception as exc:  # defensive: classifier must never break routing
        logger.warning("Stage-2 classifier path failed, using heuristic: %s", exc)
    return heuristic_category


def predicted_local_success(prompt):
    """Passthrough to the classifier's predicted-local-success probability, or
    None when no success head is trained yet (blocked on Phase 5 harness
    labels). Provided so the escalation gate can consult it once the head
    exists; currently INERT (returns None) and does not alter escalation. See
    docs/phase6-classifier.md."""
    try:
        import classifier_router

        return classifier_router.predict_success(prompt)
    except Exception:
        return None


In [ ]:
%%writefile confidence.py
"""Escalation gate: decide whether a local answer is likely to fail the
LLM-as-judge check and should be re-answered remotely.

Design principle (per CLAUDE.md's scoring model): the accuracy gate matters
far more than token cost, so this module is deliberately conservative --
when a category has no cheap verifiable check, it defaults to NOT
escalating only when the local answer looks healthy, and always escalates
on any sign the local answer is degenerate.

Verifiable checks (cheap, local, no LLM call) are the primary signal, per
the literature-backed guidance that raw LLM self-confidence is poorly
calibrated. For those categories, difficulty is ignored entirely: a
structurally valid answer is trusted even on "hard"-labeled prompts
(project memory recorded the local 3B solving a "hard" math question).

For the NO-CHECK categories (factual, summarization, code-debug, logic),
the 2026-07-10 live sweep changed the policy: hard difficulty alone now
escalates. The sweep measured the old hard-AND-hedge gate leaking a
confident hallucination, while escalating all hard no-check tasks bought
+1.7% accuracy for ~858 tokens -- cheap insurance given that failing the
accuracy gate is catastrophic (zero score) while extra tokens are only a
marginal ranking penalty.

The math check is no longer format-only: callers may pass a second,
independently generated local answer (verify_answer) and the check fails
when the two final numbers disagree -- a zero-scored-token dual-answer
agreement verifier (local tokens are free). The first live sweep measured
the format-only check leaking 2/20 wrong math answers.
"""

import ast
import json
import re

_SENTIMENT_LABELS = {"positive", "negative", "neutral"}
_NUMBER_RE = re.compile(r"-?\d+(?:\.\d+)?")
_CODE_FENCE_RE = re.compile(r"^```[a-zA-Z0-9]*\s*|\s*```$", re.M)
_DEGENERATE_ANSWERS = {"", "n/a", "na"}
_HEDGE_RE = re.compile(
    r"\b(i'?m not sure|i don'?t know|not certain|unable to determine|"
    r"cannot determine|unclear|i cannot answer|no information|"
    r"insufficient information|as an ai)\b",
    re.I,
)

# Categories with a cheap, local, no-LLM-call verifiable check.
_VERIFIABLE_CATEGORIES = {"math", "ner", "sentiment", "code-gen"}


def _looks_degenerate(answer):
    """Catches local_model.py's own failure fallback ("N/A") and any
    empty/blank answer -- a strong, category-independent failure signal
    that should escalate regardless of verifiable checks or difficulty."""
    return (answer or "").strip().lower() in _DEGENERATE_ANSWERS


def _has_hedge_language(answer):
    return bool(_HEDGE_RE.search(answer or ""))


# Same tolerances as eval/metrics.py's numeric comparison.
_NUM_REL_TOL = 1e-3
_NUM_ABS_TOL = 1e-6


def _final_number(text):
    """The LAST number in the text (the per-category system prompt asks for
    the final answer on its own line, so last = final), or None."""
    matches = _NUMBER_RE.findall(text or "")
    if not matches:
        return None
    try:
        return float(matches[-1])
    except ValueError:
        return None


def _num_close(a, b):
    return abs(a - b) <= max(_NUM_ABS_TOL, _NUM_REL_TOL * max(abs(a), abs(b)))


def _check_math(answer, verify_answer=None):
    """Format check (a parseable number must be present) plus, when a second
    independently generated answer is supplied, dual-answer agreement: the
    final numbers of both answers must match. Disagreement -- or a verify
    answer with no number at all -- fails the check (conservative: a failed
    recompute is itself a failure signal). verify_answer=None preserves the
    old format-only behavior for callers without a second answer."""
    final = _final_number(answer)
    if final is None:
        return False
    if verify_answer is not None:
        verify_final = _final_number(verify_answer)
        if verify_final is None or not _num_close(final, verify_final):
            return False
    return True


def _check_ner(answer):
    """Passes if the answer is a valid JSON list. Only falls back to
    extracting a bracketed substring when the full text fails to parse at
    all (e.g. "Here you go: [...]" prose wrapping) -- if the full text
    parses successfully to something other than a list (e.g. a JSON
    object), that result is trusted and NOT overridden by a nested
    list found inside it."""
    text = (answer or "").strip()
    try:
        parsed = json.loads(text)
    except (json.JSONDecodeError, ValueError):
        pass
    else:
        return isinstance(parsed, list)

    start, end = text.find("["), text.rfind("]")
    if start != -1 and end != -1 and end > start:
        try:
            return isinstance(json.loads(text[start : end + 1]), list)
        except (json.JSONDecodeError, ValueError):
            return False
    return False


def _check_sentiment(answer):
    """Passes if the answer is exactly one of the allowed labels, modulo
    surrounding punctuation/whitespace/case."""
    cleaned = re.sub(r"[^a-zA-Z]", "", (answer or "")).lower()
    return cleaned in _SENTIMENT_LABELS


def _check_code_gen(answer):
    """Passes if the answer parses as valid Python via ast.parse, after
    stripping markdown code fences. Best-effort: any parse failure (or a
    non-Python answer) counts as a failed check, never crashes."""
    text = _CODE_FENCE_RE.sub("", (answer or "")).strip()
    if not text:
        return False
    try:
        ast.parse(text)
        return True
    except SyntaxError:
        return False
    except Exception:
        return False


_VERIFIABLE_CHECKS = {
    "math": _check_math,
    "ner": _check_ner,
    "sentiment": _check_sentiment,
    "code-gen": _check_code_gen,
}


def should_escalate(category, difficulty, local_answer, verify_answer=None):
    """Return True if the local answer should be replaced with a remote
    (Fireworks) answer.

    verify_answer (optional): a second, independently generated LOCAL answer
    for math tasks; when provided, the math check additionally requires the
    two final numbers to agree. Zero scored-token cost (local tokens are
    free). Ignored for every other category.

    Precedence:
    1. Degenerate local answer (empty/N/A) -> always escalate.
    2. Category has a verifiable check -> escalate iff it fails. Difficulty
       is NOT consulted here: a structurally valid answer (e.g. an agreeing
       number for math, a JSON list for NER) is trusted regardless of the
       heuristic router's difficulty label.
    3. No verifiable check exists for this category (factual,
       summarization, code-debug, logic) -> escalate when difficulty is
       "hard" OR the answer shows hedging/uncertainty language. (Live-sweep
       tightening, 2026-07-10: the old hard-AND-hedge rule leaked a
       confident hallucination; hard-alone escalation on no-check
       categories measured +1.7% accuracy for ~858 tokens.)
    """
    if _looks_degenerate(local_answer):
        return True

    if category == "math":
        return not _check_math(local_answer, verify_answer)

    check = _VERIFIABLE_CHECKS.get(category)
    if check is not None:
        return not check(local_answer)

    if difficulty == "hard" or _has_hedge_language(local_answer):
        return True

    return False


In [ ]:
import os
import sys

sys.path.insert(0, ".")

from llama_cpp import Llama

from confidence import _check_math, should_escalate
from router import route_task

llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_threads=os.cpu_count(), verbose=False)

VERIFY_PREFIX = (
    "Recompute carefully step by step, then give the final numeric answer "
    "on its own line.\n\n"
)


def local_chat(system, user, max_tokens, stop=None, temperature=0.0):
    """Same contract as the project's src/local_model.py: never raises,
    returns (text, usage=None)."""
    try:
        response = llm.create_chat_completion(
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            max_tokens=max_tokens,
            temperature=temperature,
            stop=stop,
        )
        text = response["choices"][0]["message"]["content"] or ""
    except Exception as e:
        print("local_chat failed:", e)
        return "N/A", None
    return text.strip(), None


def route_and_answer(prompt):
    category, category_name, difficulty = route_task(prompt)
    answer, _ = local_chat(
        category.system_prompt, prompt, max_tokens=category.max_tokens, stop=category.stop
    )

    verify_answer = None
    verify_note = ""
    if category_name == "math":
        verify_answer, _ = local_chat(
            category.system_prompt,
            VERIFY_PREFIX + prompt,
            max_tokens=category.max_tokens,
            stop=category.stop,
        )
        agree = _check_math(answer, verify_answer)
        verify_note = (
            f"dual-answer verify: {verify_answer!r} -- "
            f"{'agrees' if agree else 'DISAGREES'} with the primary answer"
        )

    escalate = should_escalate(category_name, difficulty, answer, verify_answer)
    verdict = (
        "ESCALATE to Fireworks (not called in this demo)" if escalate else "STAYS LOCAL"
    )

    print(f"category:   {category_name}")
    print(f"difficulty: {difficulty}")
    print(f"answer:     {answer}")
    if verify_note:
        print(f"            {verify_note}")
    print(f"verdict:    {verdict}")
    return category_name, difficulty, answer, escalate


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

EXAMPLES = [
    "What is the capital of Australia?",
    "A store had 128 apples. It sold 47 in the morning and 36 in the afternoon. "
    "How many apples are left?",
    "The review says: 'This laptop is a disaster, it overheats constantly and "
    "the battery dies in an hour.' What is the sentiment?",
    "Summarize: The city council voted 5-2 Tuesday to approve a $40 million "
    "bond for a new public library after two years of debate over the site "
    "and funding sources.",
    "Extract all named entities from: 'Marie Curie won the Nobel Prize in "
    "Physics in 1903 while working in Paris.'",
    "Here's my code, it has a bug:\n```python\ndef add(a, b):\n    return a - b\n```",
    "All cats are mammals. Some mammals can fly. Can we deduce that some cats "
    "can fly? Answer yes or no and explain briefly.",
    "Write a Python function that returns the nth Fibonacci number.",
]

prompt_box = widgets.Textarea(
    placeholder="Type a prompt...", layout=widgets.Layout(width="100%", height="80px")
)
example_dropdown = widgets.Dropdown(
    options=[("-- pick an example --", "")]
    + [(p[:60] + ("..." if len(p) > 60 else ""), p) for p in EXAMPLES],
    description="Examples:",
)
run_button = widgets.Button(description="Route & Answer", button_style="primary")
output = widgets.Output()


def _on_example_change(change):
    if change["new"]:
        prompt_box.value = change["new"]


def _on_run_click(_):
    with output:
        clear_output()
        if not prompt_box.value.strip():
            print("Enter a prompt first.")
            return
        route_and_answer(prompt_box.value)


example_dropdown.observe(_on_example_change, names="value")
run_button.on_click(_on_run_click)

display(example_dropdown, prompt_box, run_button, output)
